# Kaggle step 2 · Train the conditional diffusion model on Kaggle

The same training as `03_train_diffusion.ipynb`, but on Kaggle: it keeps running in the
background ("Save Version → Save & Run All"), even if your laptop sleeps.

**Settings (right panel):** Accelerator **GPU T4 x2**, Internet **On** (needs a verified phone
number). Paste the Drive share link of `tiles.zip` below, then Save Version → Save & Run All.

**After it finishes (about 3–4 h):** open the run's **Output** tab and download the 4 files in
`checkpoints_diffusion/` (`best_ema.pt`, `stats.json`, `history.csv`, `config.json`). Upload
them to Google Drive in `MyDrive/kerala_hazard_vision_ai/checkpoints_diffusion/`.

In [ ]:
TILES_URL = "PASTE_THE_DRIVE_SHARE_LINK_OF_tiles.zip_HERE"
REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/kaggle/working/kerala-hazard-vision-ai"

!git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q . gdown

In [ ]:
import dataclasses
import logging
import shutil
import zipfile
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import pandas as pd
import torch

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.train_diffusion import fit_diffusion

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU: enable GPU T4 x2")

# Tiles go to /tmp: only /kaggle/working is saved as output, and it should hold just the model.
TILE_DIR = Path("/tmp/tiles")
OUT_DIR = Path("/kaggle/working/checkpoints_diffusion")
zip_path = Path("/tmp/tiles.zip")
gdown.download(TILES_URL, str(zip_path), fuzzy=True, quiet=False)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(TILE_DIR)
zip_path.unlink()
print(f"{len(list(TILE_DIR.glob('*.npz')))} tiles ready")

## Training

In [ ]:
cfg = load_config("configs/default.yaml")
run_cfg = dataclasses.replace(
    cfg, diffusion=dataclasses.replace(cfg.diffusion, checkpoint_dir=str(OUT_DIR))
)
fit_diffusion(run_cfg, TILE_DIR, resume=False)

# The full training state (last.pt, ~0.5 GB) is not needed for evaluation: keep it out of the
# output so the download stays small.
shutil.move(OUT_DIR / "last.pt", "/tmp/last.pt")
sorted(p.name for p in OUT_DIR.iterdir())

## Loss curves

In [ ]:
history = pd.read_csv(OUT_DIR / "history.csv")
ax = history.plot(x="epoch", y=["train_loss", "val_loss"], logy=True, figsize=(6, 4),
                  title="Diffusion denoising loss (MSE on noise)")
ax.figure.savefig(OUT_DIR / "losses_diffusion.png", dpi=150)
plt.show()
history.tail()